In [2]:
import os
import copy
import random
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models


# =========================================================
# CONFIG
# =========================================================
class CFG:
    DATA_ROOT = "/kaggle/input/datasets/seifelmedany/ct-images/CT"   # change this
    IMG_SIZE = 224
    BATCH_SIZE = 16
    EPOCHS = 2
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 2
    SEED = 42
    VAL_SIZE = 0.2
    USE_WEIGHTED_SAMPLER = True
    EARLY_STOPPING_PATIENCE = 5

    MODEL_PATH = "/kaggle/working/best_ct_binary_classifier.pth"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================================================
# LABELS
# =========================================================
LABEL_TO_INDEX = {
    "cancer": 0,
    "normal": 1
}
INDEX_TO_LABEL = {v: k for k, v in LABEL_TO_INDEX.items()}

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


# =========================================================
# REPRODUCIBILITY
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# DATA COLLECTION
# =========================================================
def collect_samples(data_root):
    data_root = Path(data_root)
    samples = []

    print(f"\nScanning dataset: {data_root}")
    print(f"Exists: {data_root.exists()}")

    for class_name in LABEL_TO_INDEX.keys():
        class_dir = data_root / class_name
        if not class_dir.exists():
            print(f"Warning: missing folder -> {class_dir}")
            continue

        image_files = sorted([p for p in class_dir.glob("*") if p.suffix.lower() in VALID_EXTENSIONS])
        print(f"{class_name}: {len(image_files)} images")

        label_idx = LABEL_TO_INDEX[class_name]
        for img_path in image_files:
            samples.append((str(img_path), label_idx))

    return samples


# =========================================================
# DATASET
# =========================================================
class CTDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


# =========================================================
# TRANSFORMS
# =========================================================
def get_transforms(img_size=224):
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    valid_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    return train_transform, valid_transform


# =========================================================
# TRAIN / VAL SPLIT
# =========================================================
def split_samples(samples, val_size=0.2, seed=42):
    labels = [label for _, label in samples]

    train_samples, val_samples = train_test_split(
        samples,
        test_size=val_size,
        random_state=seed,
        stratify=labels
    )

    return train_samples, val_samples


# =========================================================
# DATALOADERS
# =========================================================
def make_loaders(train_samples, val_samples):
    train_tfms, valid_tfms = get_transforms(CFG.IMG_SIZE)

    train_ds = CTDataset(train_samples, transform=train_tfms)
    val_ds = CTDataset(val_samples, transform=valid_tfms)

    if CFG.USE_WEIGHTED_SAMPLER:
        train_labels = [label for _, label in train_samples]
        class_count = np.bincount(train_labels, minlength=len(LABEL_TO_INDEX))
        class_weights = 1.0 / np.maximum(class_count, 1)
        sample_weights = [class_weights[label] for label in train_labels]

        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=CFG.BATCH_SIZE,
            sampler=sampler,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=True
        )
    else:
        train_loader = DataLoader(
            train_ds,
            batch_size=CFG.BATCH_SIZE,
            shuffle=True,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=True
        )

    val_loader = DataLoader(
        val_ds,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=True
    )

    return train_loader, val_loader


# =========================================================
# MODEL
# =========================================================
def build_model(num_classes=2):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


# =========================================================
# METRICS
# =========================================================
def compute_metrics(labels_all, preds_all):
    return {
        "acc": accuracy_score(labels_all, preds_all),
        "f1": f1_score(labels_all, preds_all, average="binary"),
        "precision": precision_score(labels_all, preds_all, average="binary", zero_division=0),
        "recall": recall_score(labels_all, preds_all, average="binary", zero_division=0),
    }


# =========================================================
# TRAIN / VALIDATE
# =========================================================
def train_one_epoch(model, loader, criterion, optimizer, device, epoch):
    model.train()
    running_loss = 0.0
    preds_all = []
    labels_all = []

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Train]", leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

        metrics = compute_metrics(labels_all, preds_all)
        pbar.set_postfix(
            loss=f"{running_loss / len(labels_all):.4f}",
            acc=f"{metrics['acc']:.4f}",
            f1=f"{metrics['f1']:.4f}"
        )

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(labels_all, preds_all)
    return epoch_loss, metrics


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, epoch):
    model.eval()
    running_loss = 0.0
    preds_all = []
    labels_all = []

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Val]", leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

        metrics = compute_metrics(labels_all, preds_all)
        pbar.set_postfix(
            loss=f"{running_loss / len(labels_all):.4f}",
            acc=f"{metrics['acc']:.4f}",
            f1=f"{metrics['f1']:.4f}"
        )

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(labels_all, preds_all)
    return epoch_loss, metrics, labels_all, preds_all


# =========================================================
# FINAL EVALUATION
# =========================================================
def evaluate_predictions(labels_all, preds_all):
    print("\nFinal Validation Accuracy:", accuracy_score(labels_all, preds_all))
    print("Final Validation F1:", f1_score(labels_all, preds_all, average="binary"))

    target_names = [INDEX_TO_LABEL[i] for i in range(len(INDEX_TO_LABEL))]
    print("\nClassification Report:")
    print(classification_report(labels_all, preds_all, target_names=target_names, zero_division=0))

    print("Confusion Matrix:")
    print(confusion_matrix(labels_all, preds_all))


# =========================================================
# SINGLE IMAGE PREDICTION
# =========================================================
@torch.no_grad()
def predict_image(model, image_path, device):
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    outputs = model(image_tensor)
    probs = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probs).item()

    pred_label = INDEX_TO_LABEL[pred_idx]
    confidence = probs[pred_idx].item()
    all_probs = {INDEX_TO_LABEL[i]: float(p) for i, p in enumerate(probs.cpu().numpy())}

    print(f"Prediction: {pred_label}")
    print(f"Confidence: {confidence:.4f}")
    print(f"All probabilities: {all_probs}")

    return pred_label, confidence, all_probs


# =========================================================
# MAIN
# =========================================================
def main():
    set_seed(CFG.SEED)
    print("Using device:", CFG.DEVICE)

    all_samples = collect_samples(CFG.DATA_ROOT)
    print(f"\nTotal samples found: {len(all_samples)}")

    if len(all_samples) == 0:
        raise ValueError("No images found. Check dataset path and folder names.")

    train_samples, val_samples = split_samples(
        all_samples,
        val_size=CFG.VAL_SIZE,
        seed=CFG.SEED
    )

    print(f"Train samples: {len(train_samples)}")
    print(f"Val samples:   {len(val_samples)}")

    train_loader, val_loader = make_loaders(train_samples, val_samples)

    model = build_model(num_classes=2).to(CFG.DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    best_f1 = 0.0
    best_weights = copy.deepcopy(model.state_dict())
    early_stop_counter = 0

    epoch_bar = tqdm(range(1, CFG.EPOCHS + 1), desc="Training Progress")

    for epoch in epoch_bar:
        train_loss, train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, CFG.DEVICE, epoch
        )

        val_loss, val_metrics, val_labels, val_preds = validate_one_epoch(
            model, val_loader, criterion, CFG.DEVICE, epoch
        )

        scheduler.step(val_metrics["f1"])

        print(
            f"\nEpoch [{epoch}/{CFG.EPOCHS}] "
            f"| Train Loss: {train_loss:.4f} | Train Acc: {train_metrics['acc']:.4f} | Train F1: {train_metrics['f1']:.4f} "
            f"| Val Loss: {val_loss:.4f} | Val Acc: {val_metrics['acc']:.4f} | Val F1: {val_metrics['f1']:.4f} "
            f"| Val Precision: {val_metrics['precision']:.4f} | Val Recall: {val_metrics['recall']:.4f}"
        )

        epoch_bar.set_postfix(best_val_f1=f"{best_f1:.4f}", current_val_f1=f"{val_metrics['f1']:.4f}")

        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, CFG.MODEL_PATH)
            print(f"Best model saved to {CFG.MODEL_PATH}")
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            print(f"No improvement. Early stop counter: {early_stop_counter}/{CFG.EARLY_STOPPING_PATIENCE}")

        if early_stop_counter >= CFG.EARLY_STOPPING_PATIENCE:
            print("\nEarly stopping triggered.")
            break

    print("\nLoading best model...")
    model.load_state_dict(best_weights)

    print("\nEvaluating best model on validation set...")
    _, _, val_labels, val_preds = validate_one_epoch(
        model, val_loader, criterion, CFG.DEVICE, epoch=999
    )
    evaluate_predictions(val_labels, val_preds)

    # Example prediction
    # sample_image = "/kaggle/input/your-dataset/CT-Images/CT/cancer/example.jpg"
    # predict_image(model, sample_image, CFG.DEVICE)


if __name__ == "__main__":
    main()

Using device: cuda

Scanning dataset: /kaggle/input/datasets/seifelmedany/ct-images/CT
Exists: True
cancer: 1500 images
normal: 1500 images

Total samples found: 3000
Train samples: 2400
Val samples:   600


Training Progress:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 1 [Train]:   0%|          | 0/150 [00:00<?, ?it/s]

Epoch 1 [Val]:   0%|          | 0/38 [00:00<?, ?it/s]


Epoch [1/2] | Train Loss: 0.2166 | Train Acc: 0.9175 | Train F1: 0.9132 | Val Loss: 0.0633 | Val Acc: 0.9767 | Val F1: 0.9767 | Val Precision: 0.9767 | Val Recall: 0.9767
Best model saved to /kaggle/working/best_ct_binary_classifier.pth


Epoch 2 [Train]:   0%|          | 0/150 [00:00<?, ?it/s]

Epoch 2 [Val]:   0%|          | 0/38 [00:00<?, ?it/s]


Epoch [2/2] | Train Loss: 0.0974 | Train Acc: 0.9667 | Train F1: 0.9658 | Val Loss: 0.0327 | Val Acc: 0.9933 | Val F1: 0.9934 | Val Precision: 0.9901 | Val Recall: 0.9967
Best model saved to /kaggle/working/best_ct_binary_classifier.pth

Loading best model...

Evaluating best model on validation set...


Epoch 999 [Val]:   0%|          | 0/38 [00:00<?, ?it/s]


Final Validation Accuracy: 0.9933333333333333
Final Validation F1: 0.9933554817275747

Classification Report:
              precision    recall  f1-score   support

      cancer       1.00      0.99      0.99       300
      normal       0.99      1.00      0.99       300

    accuracy                           0.99       600
   macro avg       0.99      0.99      0.99       600
weighted avg       0.99      0.99      0.99       600

Confusion Matrix:
[[297   3]
 [  1 299]]


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install -U --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 308.7 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 335.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 343.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 324.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 297.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 331.7 MB/s eta 0:00:00
     ━━━━━━━━━

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("darsh22blc1378/abdominal-ultrasound-images")

print("Path to dataset files:", path)

In [1]:
print("hellow")

hellow
